# 상품 메타데이터 분석 (고태원)
**담당 파일:** `products.csv`, `aisles.csv`, `departments.csv`  
**목표:** 상품군별로 어떤 특징이 있는가?  
**최종 산출물:** 부서별 상품 비중 차트, 인기 카테고리 TOP 10 리스트

## 1. 라이브러리 로드 및 데이터 불러오기

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 데이터 경로
DATA_DIR = '../data/'

# 데이터 로드
products    = pd.read_csv(DATA_DIR + 'products.csv')
aisles      = pd.read_csv(DATA_DIR + 'aisles.csv')
departments = pd.read_csv(DATA_DIR + 'departments.csv')
order_products = pd.read_csv(DATA_DIR + 'order_products__prior.csv')

print('products    :', products.shape)
print('aisles      :', aisles.shape)
print('departments :', departments.shape)
print('order_products:', order_products.shape)

## 2. 데이터 기본 확인

In [ ]:
display(products.head())
display(aisles.head())
display(departments.head())

## 3. 데이터 병합 (products + aisles + departments)

In [ ]:
# products에 aisle명, department명 붙이기
df = (products
      .merge(aisles, on='aisle_id')
      .merge(departments, on='department_id'))

print('병합 결과:', df.shape)
display(df.head())

## 4. 부서(Department)별 상품 수 분석
### 4-1. 부서별 상품 수 TOP 10

In [ ]:
dept_product_count = (
    df.groupby('department')['product_id']
    .count()
    .reset_index()
    .rename(columns={'product_id': 'product_count'})
    .sort_values('product_count', ascending=False)
)

print('=== 부서별 상품 수 (전체) ===')
display(dept_product_count)

In [ ]:
top10_dept = dept_product_count.head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- 왼쪽: 가로 막대 차트 (TOP 10) ---
ax1 = axes[0]
colors = sns.color_palette('Blues_d', len(top10_dept))
bars = ax1.barh(
    top10_dept['department'][::-1],
    top10_dept['product_count'][::-1],
    color=colors
)
for bar, val in zip(bars, top10_dept['product_count'][::-1]):
    ax1.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
             f'{val:,}', va='center', fontsize=10)
ax1.set_title('부서별 상품 수 TOP 10', fontsize=14, fontweight='bold')
ax1.set_xlabel('상품 수')
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax1.grid(axis='x', alpha=0.3)

# --- 오른쪽: 파이 차트 (전체 부서 비중) ---
ax2 = axes[1]
wedge_props = dict(width=0.5)  # 도넛 형태
ax2.pie(
    dept_product_count['product_count'],
    labels=dept_product_count['department'],
    autopct='%1.1f%%',
    startangle=140,
    textprops={'fontsize': 8},
    wedgeprops=wedge_props
)
ax2.set_title('부서별 상품 비중 (전체)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/dept_product_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('차트 저장 완료: data/dept_product_distribution.png')

## 5. 통로(Aisle)별 상품 수 TOP 10

In [ ]:
aisle_product_count = (
    df.groupby(['aisle', 'department'])['product_id']
    .count()
    .reset_index()
    .rename(columns={'product_id': 'product_count'})
    .sort_values('product_count', ascending=False)
)

top10_aisle = aisle_product_count.head(10)

print('=== 통로별 상품 수 TOP 10 ===')
display(top10_aisle.reset_index(drop=True))

fig, ax = plt.subplots(figsize=(10, 6))
palette = sns.color_palette('Greens_d', len(top10_aisle))
bars = ax.barh(
    top10_aisle['aisle'][::-1],
    top10_aisle['product_count'][::-1],
    color=palette
)
for bar, val, dept in zip(bars,
                           top10_aisle['product_count'][::-1],
                           top10_aisle['department'][::-1]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{val:,}  ({dept})', va='center', fontsize=9)

ax.set_title('통로(Aisle)별 상품 수 TOP 10', fontsize=14, fontweight='bold')
ax.set_xlabel('상품 수')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../data/aisle_product_top10.png', dpi=150, bbox_inches='tight')
plt.show()
print('차트 저장 완료: data/aisle_product_top10.png')

## 6. 어떤 부서의 상품이 실제 구매로 가장 많이 이어지는가?
`order_products__prior.csv`와 병합하여 실제 주문 건수 기준으로 분석합니다.

In [ ]:
# order_products에 부서/통로 정보 붙이기
order_with_meta = order_products.merge(
    df[['product_id', 'aisle', 'department']], on='product_id'
)

# 부서별 실제 주문 건수
dept_order_count = (
    order_with_meta.groupby('department')
    .size()
    .reset_index(name='order_count')
    .sort_values('order_count', ascending=False)
)

# 상품 수 대비 주문 건수 비율 (상품 1개당 평균 주문 수)
dept_combined = dept_order_count.merge(dept_product_count, on='department')
dept_combined['orders_per_product'] = (
    dept_combined['order_count'] / dept_combined['product_count']
).round(1)

print('=== 부서별 실제 주문 건수 ===')
display(dept_combined.sort_values('order_count', ascending=False).reset_index(drop=True))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- 왼쪽: 실제 주문 건수 TOP 10 ---
top10_order = dept_combined.sort_values('order_count', ascending=False).head(10)
ax1 = axes[0]
colors1 = sns.color_palette('Oranges_d', len(top10_order))
bars1 = ax1.barh(
    top10_order['department'][::-1],
    top10_order['order_count'][::-1],
    color=colors1
)
for bar, val in zip(bars1, top10_order['order_count'][::-1]):
    ax1.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
             f'{val:,}', va='center', fontsize=9)
ax1.set_title('부서별 실제 주문 건수 TOP 10', fontsize=13, fontweight='bold')
ax1.set_xlabel('주문 건수')
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x/1e6):.0f}M'))
ax1.grid(axis='x', alpha=0.3)

# --- 오른쪽: 상품 1개당 평균 주문 수 (상품 효율성) ---
top10_eff = dept_combined.sort_values('orders_per_product', ascending=False).head(10)
ax2 = axes[1]
colors2 = sns.color_palette('Purples_d', len(top10_eff))
bars2 = ax2.barh(
    top10_eff['department'][::-1],
    top10_eff['orders_per_product'][::-1],
    color=colors2
)
for bar, val in zip(bars2, top10_eff['orders_per_product'][::-1]):
    ax2.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
             f'{val:,.0f}', va='center', fontsize=9)
ax2.set_title('상품 1개당 평균 주문 수 TOP 10\n(구매 전환 효율성)', fontsize=13, fontweight='bold')
ax2.set_xlabel('평균 주문 수 / 상품')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/dept_order_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('차트 저장 완료: data/dept_order_analysis.png')

## 7. 인기 카테고리(Aisle) TOP 10 — 실제 주문 기준

In [ ]:
aisle_order_count = (
    order_with_meta.groupby(['aisle', 'department'])
    .size()
    .reset_index(name='order_count')
    .sort_values('order_count', ascending=False)
)

top10_aisle_order = aisle_order_count.head(10).reset_index(drop=True)
top10_aisle_order.index += 1  # 순위 1부터 시작

print('=== 인기 카테고리(Aisle) TOP 10 (실제 주문 건수 기준) ===')
display(top10_aisle_order)

fig, ax = plt.subplots(figsize=(11, 6))
palette = sns.color_palette('RdYlGn_r', len(top10_aisle_order))
bars = ax.barh(
    top10_aisle_order['aisle'][::-1],
    top10_aisle_order['order_count'][::-1],
    color=palette[::-1]
)
for bar, val, dept in zip(bars,
                           top10_aisle_order['order_count'][::-1],
                           top10_aisle_order['department'][::-1]):
    ax.text(bar.get_width() * 1.005, bar.get_y() + bar.get_height()/2,
            f'{val:,}  [{dept}]', va='center', fontsize=9)

ax.set_title('인기 카테고리(Aisle) TOP 10\n(실제 주문 건수 기준)', fontsize=14, fontweight='bold')
ax.set_xlabel('주문 건수')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x/1e6):.0f}M'))
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../data/aisle_popular_top10.png', dpi=150, bbox_inches='tight')
plt.show()
print('차트 저장 완료: data/aisle_popular_top10.png')

## 8. 최종 요약 리포트

In [ ]:
print('=' * 60)
print('           상품 메타데이터 분석 요약 리포트')
print('=' * 60)

print(f"\n[기본 현황]")
print(f"  총 상품 수       : {len(products):,}개")
print(f"  총 부서(dept) 수 : {len(departments):,}개")
print(f"  총 통로(aisle) 수: {len(aisles):,}개")

print(f"\n[상품 수 기준 TOP 3 부서]")
for i, row in dept_product_count.head(3).iterrows():
    print(f"  {row['department']:<20} {row['product_count']:,}개")

print(f"\n[상품 수 기준 TOP 3 통로]")
for i, row in aisle_product_count.head(3).iterrows():
    print(f"  {row['aisle']:<30} {row['product_count']:,}개  ({row['department']})")

print(f"\n[실제 주문 건수 기준 TOP 3 부서]")
for i, row in dept_combined.sort_values('order_count', ascending=False).head(3).iterrows():
    print(f"  {row['department']:<20} {row['order_count']:,}건")

print(f"\n[인기 카테고리 TOP 3 (실제 주문 기준)]")
for i, row in aisle_order_count.head(3).iterrows():
    print(f"  {row['aisle']:<30} {row['order_count']:,}건  [{row['department']}]")

print('\n' + '=' * 60)
print('[인사이트]')
print('  - 상품 수가 가장 많은 부서와 실제 주문이 많은 부서가')
print('    반드시 일치하지는 않음 → 구매 전환 효율 차이 존재')
print('  - produce(신선식품) / dairy eggs(유제품) 카테고리가')
print('    상품 수는 적어도 주문 비중이 높아 핵심 카테고리임')
print('  - personal care / international 등은 상품 수 대비')
print('    주문 전환율이 낮아 정비 또는 프로모션 필요')
print('=' * 60)